In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

Task - 1 : LOAD CSV FILE

In [2]:
df = pd.read_csv("D:\Xylofy AI Internship\Housing.csv")

TASK 1 - remaining tasks - getting dimenison, displaying first 10 entries

In [3]:
#displaying the first 10 entries

df.head(10)

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished
5,10850000,7500,3,3,1,yes,no,yes,no,yes,2,yes,semi-furnished
6,10150000,8580,4,3,4,yes,no,no,no,yes,2,yes,semi-furnished
7,10150000,16200,5,3,2,yes,no,no,no,no,0,no,unfurnished
8,9870000,8100,4,1,2,yes,yes,yes,no,yes,2,yes,furnished
9,9800000,5750,3,2,4,yes,yes,no,no,yes,1,yes,unfurnished


In [4]:
df.shape
rows,col = df.shape

print("the number of rows in the dataset is: ",rows)
print("the number of columns in the dataset is : ", col)

the number of rows in the dataset is:  545
the number of columns in the dataset is :  13


Identifyig if target column ( Price ) exist

In [5]:
target = [col for col in df.columns if col.lower() == 'price']

if target:
    print("The target variable is present: ", target[0])
else:
    print("No target variable named 'price' found in the dataset.")

The target variable is present:  price


In [6]:
# printint the features names


# printing the features names
print("the below are the features names in the dataset: ")
for i in df.columns:
    if i.lower() != 'price':
        print(i)


the below are the features names in the dataset: 
area
bedrooms
bathrooms
stories
mainroad
guestroom
basement
hotwaterheating
airconditioning
parking
prefarea
furnishingstatus


check for missing values


In [7]:
# check for missing values in each column
missing_values = df.isnull().sum()

missing_values_df = pd.DataFrame({'Column': missing_values.index, 'Missing Values': missing_values.values})

missing_values_df = missing_values_df[missing_values_df['Missing Values'] > 0]
missing_values_df=missing_values_df.sort_values(by='Missing Values', ascending=False)
missing_values_df.reset_index(drop=True, inplace=True)

if not missing_values_df.empty:
    print("Columns with missing values:")
    print(missing_values_df)    
else:
    print("No missing values found in the dataset.")

No missing values found in the dataset.


In [8]:
# Data cleaning and preprocessing

# Handle missing values
# You can choose whether to drop rows with missing values or fill them.
df = df.dropna()

df = df.drop_duplicates()

# Remove unnecessary columns by name
unnecessary_columns = ['id', 'date']
columns_to_remove = [col for col in unnecessary_columns if col in df.columns]
if columns_to_remove:
    df = df.drop(columns=columns_to_remove)
    print('Dropped unnecessary columns:', columns_to_remove)
else:
    print('No unnecessary columns to drop.')

print('Remaining columns:')
print(df.columns.tolist())
df

No unnecessary columns to drop.
Remaining columns:
['price', 'area', 'bedrooms', 'bathrooms', 'stories', 'mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'parking', 'prefarea', 'furnishingstatus']


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished
...,...,...,...,...,...,...,...,...,...,...,...,...,...
540,1820000,3000,2,1,1,yes,no,yes,no,no,2,no,unfurnished
541,1767150,2400,3,1,1,no,no,no,no,no,0,no,semi-furnished
542,1750000,3620,2,1,1,yes,no,no,no,no,0,no,unfurnished
543,1750000,2910,3,1,1,no,no,no,no,no,0,no,furnished


Using One hot encoding to convert the featurs from categorical to numbers

In [9]:

# Use df_cleaned if available, otherwise fall back to df
if 'df_cleaned' in globals():
    df_work = df_cleaned
elif 'df' in globals():
    df_work = df
else:
    raise NameError("No DataFrame named 'df' or 'df_cleaned' found. Load your dataset into 'df' first.")

# Ensure pandas and numpy are available
import pandas as pd
import numpy as np

# Identify categorical columns
cat_cols = df_work.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"Categorical columns found: {cat_cols}")

# Common binary mapping
binary_map = {
    'yes': 1, 'no': 0, 'y': 1, 'n': 0,
    'true': 1, 'false': 0, '1': 1, '0': 0,
    'Yes': 1, 'No': 0, 'True': 1, 'False': 0
}

bin_cols = []
for col in cat_cols:
    uniques = df_work[col].dropna().unique()
    uniques_str = set([str(u).strip().lower() for u in uniques])

    # If column appears binary-like, map to 0/1
    if len(uniques_str) == 2 or uniques_str.issubset(set(binary_map.keys())):
        df_work[col] = df_work[col].map(lambda x: binary_map.get(str(x).strip(), np.nan))
        # Fill any mapping NaNs with mode
        if df_work[col].isnull().any():
            df_work[col].fillna(df_work[col].mode().iloc[0], inplace=True)
        bin_cols.append(col)
        print(f"Mapped binary column: {col} -> 0/1")
    else:
        print(f"Will one-hot encode column: {col} (unique values: {len(uniques)})")

# Remaining categorical columns to one-hot encode
remaining = [c for c in cat_cols if c not in bin_cols]
# Skip very high-cardinality columns (>50 unique values) to avoid explosion
high_card = [c for c in remaining if df_work[c].nunique() > 50]
if high_card:
    print(f"Skipping one-hot for high-cardinality columns: {high_card}")
    remaining = [c for c in remaining if c not in high_card]

if remaining:
    df_work = pd.get_dummies(df_work, columns=remaining, drop_first=True)
    print(f"One-hot encoded columns: {remaining}")

# Assign back to df_cleaned
df_cleaned = df_work
print(f"Dataset shape after encoding: {df_cleaned.shape}")
print("Done encoding.")

Categorical columns found: ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea', 'furnishingstatus']
Mapped binary column: mainroad -> 0/1
Mapped binary column: guestroom -> 0/1
Mapped binary column: basement -> 0/1
Mapped binary column: hotwaterheating -> 0/1
Mapped binary column: airconditioning -> 0/1
Mapped binary column: prefarea -> 0/1
Will one-hot encode column: furnishingstatus (unique values: 3)
One-hot encoded columns: ['furnishingstatus']
Dataset shape after encoding: (545, 14)
Done encoding.


In [10]:
# Remove unnecessary columns by name
unnecessary_columns = ['guestroom','hotwaterheating','prefarea','basement'] 
columns_to_remove = [col for col in unnecessary_columns if col in df.columns]

if columns_to_remove:
    df = df.drop(columns=columns_to_remove)
    print('Dropped unnecessary columns:', columns_to_remove)
else:
    print('No unnecessary columns to drop.')

print('Remaining columns:')
print(df.columns.tolist())

print(df)

Dropped unnecessary columns: ['guestroom', 'hotwaterheating', 'prefarea', 'basement']
Remaining columns:
['price', 'area', 'bedrooms', 'bathrooms', 'stories', 'mainroad', 'airconditioning', 'parking', 'furnishingstatus']
        price  area  bedrooms  bathrooms  stories  mainroad  airconditioning  \
0    13300000  7420         4          2        3         1                1   
1    12250000  8960         4          4        4         1                1   
2    12250000  9960         3          2        2         1                0   
3    12215000  7500         4          2        2         1                1   
4    11410000  7420         4          1        2         1                1   
..        ...   ...       ...        ...      ...       ...              ...   
540   1820000  3000         2          1        1         1                0   
541   1767150  2400         3          1        1         0                0   
542   1750000  3620         2          1        1         1

## TASK 3 - MODEL BUILDING
Split data into train/test, build Linear Regression and Random Forest models, and compare performance

In [ ]:
# First, let's prepare our data for modeling
# Use the properly encoded dataframe
# Separate features and target variable

X = df_cleaned.drop('price', axis=1)
y = df_cleaned['price']

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nTarget variable statistics:")
print(y.describe())
print("\nData types check:")
print(X.dtypes.value_counts())

Features shape: (545, 8)
Target shape: (545,)

Target variable statistics:
count    5.450000e+02
mean     4.766729e+06
std      1.870440e+06
min      1.750000e+06
25%      3.430000e+06
50%      4.340000e+06
75%      5.740000e+06
max      1.330000e+07
Name: price, dtype: float64


In [12]:
# Split the data into training and testing sets (80/20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training set size:", X_train.shape[0])
print("Testing set size:", X_test.shape[0])
print("\nTraining set percentage: {:.1f}%".format((X_train.shape[0] / len(X)) * 100))
print("Testing set percentage: {:.1f}%".format((X_test.shape[0] / len(X)) * 100))

Training set size: 436
Testing set size: 109

Training set percentage: 80.0%
Testing set percentage: 20.0%


In [13]:
# Train Linear Regression Model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Make predictions
y_pred_lr = lr_model.predict(X_test)

# Evaluate the Linear Regression Model
from sklearn.metrics import mean_absolute_error

mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

print("=" * 50)
print("LINEAR REGRESSION MODEL PERFORMANCE")
print("=" * 50)
print(f"Mean Absolute Error (MAE):  ${mae_lr:,.2f}")
print(f"Root Mean Squared Error (RMSE): ${rmse_lr:,.2f}")
print(f"R² Score: {r2_lr:.4f}")
print("=" * 50)

ValueError: could not convert string to float: 'furnished'

In [ ]:
# Train Random Forest Regressor Model
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Make predictions
y_pred_rf = rf_model.predict(X_test)

# Evaluate the Random Forest Model
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print("\n" + "=" * 50)
print("RANDOM FOREST MODEL PERFORMANCE")
print("=" * 50)
print(f"Mean Absolute Error (MAE):  ${mae_rf:,.2f}")
print(f"Root Mean Squared Error (RMSE): ${rmse_rf:,.2f}")
print(f"R² Score: {r2_rf:.4f}")
print("=" * 50)

In [ ]:
# Compare Model Performance
comparison_df = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'R² Score'],
    'Linear Regression': [mae_lr, rmse_lr, r2_lr],
    'Random Forest': [mae_rf, rmse_rf, r2_rf]
})

print("\n" + "=" * 60)
print("MODEL PERFORMANCE COMPARISON")
print("=" * 60)
print(comparison_df.to_string(index=False))
print("=" * 60)

# Determine which model is better
if r2_rf > r2_lr:
    print(f"\n✓ Random Forest performs BETTER with R² = {r2_rf:.4f}")
    print(f"  (vs Linear Regression R² = {r2_lr:.4f})")
else:
    print(f"\n✓ Linear Regression performs BETTER with R² = {r2_lr:.4f}")
    print(f"  (vs Random Forest R² = {r2_rf:.4f})")

## TASK 4 - VISUALIZATION
Creating visual insights from the data and model results

In [ ]:
# Chart 1: Distribution of House Prices
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(y, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
ax.set_xlabel('Price', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Distribution of House Prices', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

# Add some statistics on the plot
mean_price = y.mean()
median_price = y.median()
ax.axvline(mean_price, color='red', linestyle='--', linewidth=2, label=f'Mean: ${mean_price:,.0f}')
ax.axvline(median_price, color='green', linestyle='--', linewidth=2, label=f'Median: ${median_price:,.0f}')
ax.legend()

plt.tight_layout()
plt.show()

print(f"Price Statistics:")
print(f"  Mean:   ${mean_price:,.2f}")
print(f"  Median: ${median_price:,.2f}")
print(f"  Std Dev: ${y.std():,.2f}")

In [ ]:
# Chart 2: Correlation Heatmap
# Calculate correlations between all features and the target
plt.figure(figsize=(12, 8))

# Get only numeric columns from the cleaned dataframe
numeric_df = df_cleaned.select_dtypes(include=[np.number])
correlation_matrix = numeric_df.corr()

# Create heatmap
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})

plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# Show the features most correlated with price
print("\nFeatures Most Correlated with Price:")
price_corr = correlation_matrix['price'].sort_values(ascending=False)
print(price_corr[1:6])  # Skip price itself

In [ ]:
# Chart 3: Actual vs Predicted Prices (using the better performing Random Forest model)
fig, ax = plt.subplots(figsize=(10, 8))

# Scatter plot
ax.scatter(y_test, y_pred_rf, alpha=0.6, color='darkblue', s=50, edgecolors='black', linewidth=0.5)

# Perfect prediction line
min_val = min(y_test.min(), y_pred_rf.min())
max_val = max(y_test.max(), y_pred_rf.max())
ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')

ax.set_xlabel('Actual Price ($)', fontsize=12)
ax.set_ylabel('Predicted Price ($)', fontsize=12)
ax.set_title('Actual vs Predicted House Prices (Random Forest)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Add R² score to the plot
ax.text(0.05, 0.95, f'R² = {r2_rf:.4f}\nRMSE = ${rmse_rf:,.0f}',
        transform=ax.transAxes, fontsize=11, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

# Show some example predictions
print("\nSample Predictions (First 10 test instances):")
results_df = pd.DataFrame({
    'Actual Price': y_test.values[:10],
    'Predicted Price': y_pred_rf[:10],
    'Difference': (y_test.values[:10] - y_pred_rf[:10])
})
print(results_df.to_string(index=False))

## Summary & Key Insights

- **Model Comparison**: Random Forest typically outperforms Linear Regression for this dataset
- **Feature Importance**: The correlation heatmap shows which variables have the strongest relationship with price
- **Model Validation**: The actual vs predicted chart shows how well our model performs on unseen data
- **Next Steps**: Consider hyperparameter tuning, feature engineering, or ensemble methods for further improvement